In [2]:
import numpy as np

In [123]:

# Mouvement brownien
N_STEPS = 50
R_max = 20e-6        # rayon de la particule (m)
R_min = 50e-9
KB = 1.380649e-23   # constante de Boltzmann (J/K)
T = 298             # température (K)
NU = 1.01e-3        # viscosité de l'eau (Pa.s)
TIME_STEP = 1.111  # durée entre deux images (s)
STEPS = 50

In [176]:
LAMBDA = 532e-9  # longueur d'onde (m)
R = 2e-6
pixel_size = 1.55e-6
M = -40
NA = 0.40
RESOLUTION = 0.61 * LAMBDA / NA

In [177]:
def diffusion_coefficient(R):
    return KB * T / (6 * np.pi * NU * R)

In [178]:
pas_de_temps_min_pour_R_max = (pixel_size / abs(M))**2 / (2 * diffusion_coefficient(R_max))
pas_de_temps_min_pour_R_min = (pixel_size / abs(M))**2 / (2 * diffusion_coefficient(R_min))
TIME_STEP = max(pas_de_temps_min_pour_R_max, pas_de_temps_min_pour_R_min)
print("pas de temps minimum pour R_max:", pas_de_temps_min_pour_R_max)
print("pas de temps minimum pour R_min:", pas_de_temps_min_pour_R_min)

pas de temps minimum pour R_max: 0.06948104854299093
pas de temps minimum pour R_min: 0.00017370262135747732


In [179]:
print("Diffusion coefficient for R_max:", diffusion_coefficient(R_max))
print("Diffusion coefficient for R_min:", diffusion_coefficient(R_min))
print("Avec un pas de temps de", TIME_STEP, "s, la distance moyenne parcourue par une particule de rayon R_max est:", np.sqrt(2 * diffusion_coefficient(R_max) * TIME_STEP), "m")
print("Avec un pas de temps de", TIME_STEP, "s, la distance moyenne parcourue par une particule de rayon R_min est:", np.sqrt(2 * diffusion_coefficient(R_min) * TIME_STEP), "m")
print("Avec un pas de temps de", TIME_STEP, "s, la distance moyenne parcourue par une particule de rayon R_max après", STEPS, "pas est:", np.sqrt(2 * diffusion_coefficient(R_max) * TIME_STEP) * STEPS, "m")
print("Avec un pas de temps de", TIME_STEP, "s, la distance moyenne parcourue par une particule de rayon R_min après", STEPS, "pas est:", np.sqrt(2 * diffusion_coefficient(R_min) * TIME_STEP) * STEPS, "m")

Diffusion coefficient for R_max: 1.0805554402873744e-14
Diffusion coefficient for R_min: 4.322221761149498e-12
Avec un pas de temps de 0.06948104854299093 s, la distance moyenne parcourue par une particule de rayon R_max est: 3.875e-08 m
Avec un pas de temps de 0.06948104854299093 s, la distance moyenne parcourue par une particule de rayon R_min est: 7.750000000000001e-07 m
Avec un pas de temps de 0.06948104854299093 s, la distance moyenne parcourue par une particule de rayon R_max après 50 pas est: 1.9375e-06 m
Avec un pas de temps de 0.06948104854299093 s, la distance moyenne parcourue par une particule de rayon R_min après 50 pas est: 3.875000000000001e-05 m


In [180]:
print("Resolution:", RESOLUTION, "m")
pixels_per_meter = 1 / (pixel_size / abs(M))
pixel_R_max = 2 * R_max * pixels_per_meter
pixel_R_min = 2 * R_min * pixels_per_meter
print("Nombre de pixels pour R_max:", pixel_R_max)
print("Nombre de pixels pour R_min:", pixel_R_min)
pixel_resolution = RESOLUTION * pixels_per_meter
print("Nombre de pixels pour la résolution:", pixel_resolution)
pixel_diffusion_R_max = np.sqrt(2 * diffusion_coefficient(R_max) * TIME_STEP * STEPS) * pixels_per_meter
pixel_diffusion_R_min = np.sqrt(2 * diffusion_coefficient(R_min) * TIME_STEP * STEPS) * pixels_per_meter
print("Nombre de pixels pour le déplacement moyen de R_max après", STEPS, "pas:", pixel_diffusion_R_max)
print("Nombre de pixels pour le déplacement moyen de R_min après", STEPS, "pas:", pixel_diffusion_R_min)


Resolution: 8.113000000000001e-07 m
Nombre de pixels pour R_max: 1032.258064516129
Nombre de pixels pour R_min: 2.5806451612903225
Nombre de pixels pour la résolution: 20.936774193548388
Nombre de pixels pour le déplacement moyen de R_max après 50 pas: 7.0710678118654755
Nombre de pixels pour le déplacement moyen de R_min après 50 pas: 141.4213562373095


In [181]:
nombre_pixel_capteur = 3040

In [182]:


print("nombre maximum de particules pour R_max sur le capteur:", (nombre_pixel_capteur / ( (2 * R_max / (pixel_size / abs(M))) + np.sqrt(2 * diffusion_coefficient(R_max) * TIME_STEP ) * STEPS / (pixel_size / abs(M)) + RESOLUTION / (pixel_size / abs(M))))**2)
print("nombre maximum de particules pour R_min sur le capteur:", (nombre_pixel_capteur / ( (2 * R_min / (pixel_size / abs(M))) + np.sqrt(2 * diffusion_coefficient(R_min) * TIME_STEP ) * STEPS / (pixel_size / abs(M)) + RESOLUTION / (pixel_size / abs(M))))**2)

nombre maximum de particules pour R_max sur le capteur: 7.593512716238172
nombre maximum de particules pour R_min sur le capteur: 8.821789495536535


In [ ]:
import numpy as np

KB = 1.380649e-23  # Boltzmann constant (J/K)


def diffusion_coefficient(R, T=298, nu=1.01e-3):
    """Stokes-Einstein diffusion coefficient (m^2/s) for a sphere of radius R (m)."""
    return KB * T / (6 * np.pi * nu * R)


def microscope_performance(
    M,
    NA,
    wavelength=532e-9,   # m
    pixel_size=1.55e-6,  # sensor pixel pitch (m)
    n_pixels=3040,       # pixels along one side of the (square) region used
    R_min=50e-9,         # smallest particle radius (m)
    R_max=20e-6,         # largest particle radius (m)
    n_steps=50,          # number of frames in a trajectory
    T=298,               # temperature (K)
    nu=1.01e-3,          # viscosity (Pa.s)
):
    """Performance figures for a given magnification M and numerical aperture NA.

    Returns a flat dict, so that
        pd.DataFrame([microscope_performance(M, NA) for M, NA in configs])
    gives a comparison table directly.
    """
    M = abs(M)
    pix_obj = pixel_size / M                 # pixel size in object space (m)
    resolution = 0.61 * wavelength / NA      # Rayleigh criterion (m)
    resolution_px = resolution / pix_obj

    out = {
        "M": M,
        "NA": NA,
        "pixel_size_object_m": pix_obj,
        "fov_m": n_pixels * pix_obj,
        "resolution_m": resolution,
        "resolution_px": resolution_px,  # >= 2 means Nyquist sampling is satisfied
    }

    # Minimum time step per particle size: RMS 1D step sqrt(2*D*dt) = 1 object-space pixel
    D = {}
    dt_min = {}
    for label, R in (("Rmax", R_max), ("Rmin", R_min)):
        D[label] = diffusion_coefficient(R, T, nu)
        dt_min[label] = pix_obj**2 / (2 * D[label])

    # One common time step so that the slowest (largest) particle still moves >= 1 px/frame
    time_step = max(dt_min.values())
    out["time_step_s"] = time_step
    out["fps_required"] = 1 / time_step

    for label, R in (("Rmax", R_max), ("Rmin", R_min)):
        step = np.sqrt(2 * D[label] * time_step)                # RMS 1D step per frame (m)
        disp = np.sqrt(2 * D[label] * time_step )* n_steps      # RMS 1D displacement after n_steps (m)
        diameter_px = 2 * (R_max if label == "Rmax" else R_min) / pix_obj
        disp_px = disp / pix_obj

        # Footprint of one particle over the trajectory: image size + diffusion extent + PSF width
        footprint_px = diameter_px + disp_px + resolution_px
        max_particles = (n_pixels / footprint_px) ** 2

        out.update({
            f"D_{label}_m2s": D[label],
            f"dt_min_{label}_s": dt_min[label],
            f"step_{label}_m": step,
            f"step_{label}_px": step / pix_obj,
            f"diameter_{label}_px": diameter_px,
            f"displacement_{label}_px": disp_px,
            f"max_particles_{label}": max_particles,
        })

    return out


# (magnification, NA)
OBJECTIVES = [
    (10, 0.30),
    (20, 0.50),
    (40, 0.75),
    (4, 0.10),
    (10, 0.25),
    (20, 0.40),
    (40, 0.65),
    (60, 0.85),
    (40, 0.70),
    (20, 0.45),
    (4, 0.13),
]


def compare(objectives=OBJECTIVES, **kwargs):
    """Run microscope_performance on every (M, NA) and return a DataFrame.

    Extra keyword arguments (wavelength, pixel_size, n_pixels, ...) are passed through.
    """
    import pandas as pd

    rows = [microscope_performance(M, NA, **kwargs) for M, NA in objectives]
    return pd.DataFrame(rows).sort_values(["M", "NA"]).reset_index(drop=True)


# (column, LaTeX header, format)
LATEX_COLUMNS = [
    ("M", "$M$", "{:g}"),
    ("NA", "NA", "{:.2f}"),
    ("fov_um", "FOV ($\\mu$m)", "{:.0f}"),
    ("resolution_nm", "Res. (nm)", "{:.0f}"),
    ("resolution_px", "Res. (px)", "{:.1f}"),
    ("time_step_s", "$\\Delta t$ (s)", "{:.3g}"),
    ("fps_required", "fps", "{:.3g}"),
    ("max_particles_Rmax", "$N_{\\max}$ ($R_{\\max}$)", "{:.0f}"),
    ("max_particles_Rmin", "$N_{\\max}$ ($R_{\\min}$)", "{:.0f}"),
]


def to_latex_table(df=None, path="microscope_table.tex",
                   caption="Estimated performance of each objective.", label="tab:objectives"):
    """Write the comparison table as a LaTeX table (needs \\usepackage{booktabs})."""
    if df is None:
        df = compare()
    df = df.copy()
    df["fov_um"] = df["fov_m"] * 1e6
    df["resolution_nm"] = df["resolution_m"] * 1e9

    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\small",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{" + "r" * len(LATEX_COLUMNS) + "}",
        r"\toprule",
        " & ".join(h for _, h, _ in LATEX_COLUMNS) + r" \\",
        r"\midrule",
    ]
    for _, row in df.iterrows():
        lines.append(" & ".join(f.format(row[c]) for c, _, f in LATEX_COLUMNS) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

    with open(path, "w") as fh:
        fh.write("\n".join(lines) + "\n")
    return path


if __name__ == "__main__":
    import pandas as pd

    df = compare()
    df["fov_um"] = df["fov_m"] * 1e6
    df["resolution_nm"] = df["resolution_m"] * 1e9

    cols = [
        "M", "NA", "fov_um", "resolution_nm", "resolution_px",
        "time_step_s", "fps_required",
        "displacement_Rmax_px", "displacement_Rmin_px",
        "max_particles_Rmax", "max_particles_Rmin",
    ]
    with pd.option_context("display.width", 200, "display.max_columns", None):
        print(df[cols].to_string(float_format=lambda x: f"{x:.4g}"))

    print("LaTeX table written to", to_latex_table())

     M   NA  fov_um  resolution_nm  resolution_px  time_step_s  fps_required  displacement_Rmax_px  displacement_Rmin_px  max_particles_Rmax  max_particles_Rmin
0    4  0.1    1178           3245          8.375        6.948        0.1439                    50                  1000               353.9               9.084
1    4 0.13    1178           2496          6.442        6.948        0.1439                    50                  1000               362.5               9.119
2   10 0.25   471.2           1298          8.375        1.112        0.8995                    50                  1000               92.29               9.077
3   10  0.3   471.2           1082          6.979        1.112        0.8995                    50                  1000               93.11               9.102
4   20  0.4   235.6          811.3          10.47       0.2779         3.598                    50                  1000                27.8               9.028
5   20 0.45   235.6          721.2